# Session 8: Train the Mood Machine

Click a cell and press **Shift + Enter** to run it. Work from top to bottom.
Cells marked **YOUR TURN** are for you to finish. Run the first cell straight away: it rebuilds the faces and MoodNet.

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
import matplotlib.pyplot as plt
from emoji_faces import make_face, CLASSES

if torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator()
else:
    device = torch.device("cpu")
print("Using", device)

class EmojiFaces(Dataset):
    def __init__(self, how_many, first_seed, transform=None):
        self.faces = []
        for seed in range(first_seed, first_seed + how_many):
            picture, mood = make_face(seed)
            if transform:
                picture = transform(picture)
            self.faces.append((picture, mood))

    def __len__(self):
        return len(self.faces)

    def __getitem__(self, idx):
        return self.faces[idx]

get_ready = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

print("Drawing faces... (about a minute)")
train_faces = EmojiFaces(12000, first_seed=0, transform=get_ready)
test_faces = EmojiFaces(3000, first_seed=1_000_000, transform=get_ready)
train_loader = DataLoader(train_faces, batch_size=64, shuffle=True)
test_loader = DataLoader(test_faces, batch_size=64)
print(len(train_faces), "training faces and", len(test_faces), "test faces")

class MoodNet(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()
        self.flatten = nn.Flatten()
        self.hidden1 = nn.Linear(3 * 32 * 32, hidden)
        self.hidden2 = nn.Linear(hidden, 128)
        self.output = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.hidden1(x))
        x = self.relu(self.hidden2(x))
        return self.output(x)

torch.manual_seed(1)
model = MoodNet().to(device)

## 1. Cross-Entropy Loss by Hand
Softmax turns scores into chances. The loss is minus the log of the chance given to the right answer.

In [ ]:
import math
for chance in [0.9, 0.5, 0.1, 0.01]:
    print("right mood given", chance, " loss", round(-math.log(chance), 2))

In [ ]:
scores = torch.tensor([[2.0, 0.5, 0.1, -1.0, 0.0, 0.3, 0.2, -0.5, 0.1, 0.0]])   # one face, 10 scores
right = torch.tensor([0])                                                          # the right mood is Happy
chances = torch.softmax(scores, dim=1)
print("chance for Happy:", round(chances[0, 0].item(), 3))
print("by hand:  ", round(-math.log(chances[0, 0].item()), 4))
print("PyTorch:  ", round(nn.CrossEntropyLoss()(scores, right).item(), 4))

In [ ]:
# YOUR TURN: change the scores so the loss is very small, then very big. What did you change?
small = torch.tensor([[2.0, 0.5, 0.1, -1.0, 0.0, 0.3, 0.2, -0.5, 0.1, 0.0]])
big = torch.tensor([[2.0, 0.5, 0.1, -1.0, 0.0, 0.3, 0.2, -0.5, 0.1, 0.0]])
print(round(nn.CrossEntropyLoss()(small, right).item(), 4), round(nn.CrossEntropyLoss()(big, right).item(), 4))

## 2. The Training Loop and the Marking Function

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimiser):
    model.train()
    for faces, moods in loader:
        faces, moods = faces.to(device), moods.to(device)
        guesses = model(faces)            # 1. make guesses
        loss = loss_fn(guesses, moods)    # 2. how wrong were they?
        optimiser.zero_grad()             # 3. wipe old gradients
        loss.backward()                   # 4. find new gradients (back propagation)
        optimiser.step()                  # 5. nudge the weights
    return loss.item()

def check_accuracy(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for faces, moods in loader:
            faces, moods = faces.to(device), moods.to(device)
            guesses = model(faces).argmax(dim=1)
            correct += (guesses == moods).sum().item()
    return 100 * correct / len(loader.dataset)

## 3. Plain Gradient Descent or Adam?
One epoch each, from the same starting weights.

In [ ]:
for name in ["plain gradient descent", "Adam"]:
    torch.manual_seed(1)
    trial = MoodNet().to(device)
    if name == "Adam":
        opt = torch.optim.Adam(trial.parameters(), lr=0.001)
    else:
        opt = torch.optim.SGD(trial.parameters(), lr=0.001)
    train_one_epoch(trial, train_loader, nn.CrossEntropyLoss(), opt)
    print(name, "after one epoch:", round(check_accuracy(trial, test_loader), 1), "%")

## 4. Train!
Five epochs take one to three minutes.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
print("before training:", round(check_accuracy(model, test_loader), 1), "%")
for epoch in range(1, 6):
    last_loss = train_one_epoch(model, train_loader, loss_fn, optimiser)
    print("epoch", epoch, " last batch loss", round(last_loss, 3),
          " test accuracy", round(check_accuracy(model, test_loader), 1), "%")

## 5. Save It

In [ ]:
torch.save(model.state_dict(), "mood_machine.pth")
print("Saved!")

## 6. Stretch: Write the Loop Yourself
Train a brand-new MoodNet for one epoch without using `train_one_epoch()`. Follow the comments, then run the CHECK cell. The hints are on the website.

In [ ]:
# WRITE IT YOURSELF: one epoch of training, from scratch.
torch.manual_seed(2)

# 1. Make a new MoodNet called my_model, and move it to the device.
my_model = ...

# 2. Make a cross-entropy loss function, and an Adam optimiser with learning rate 0.001.


# 3. Put my_model into training mode.


# 4. For every batch of faces and moods in train_loader:
#    a. move the faces and moods to the device
#    b. make guesses
#    c. work out the loss
#    d. wipe the old gradients
#    e. find new gradients
#    f. nudge the weights

In [ ]:
# CHECK: run this to test your loop
try:
    score = check_accuracy(my_model, test_loader)
    if score > 60:
        print("It works! Your Mood Machine scored", round(score, 1), "% after one epoch.")
    else:
        print("It ran, but only scored", round(score, 1), "%. Did you call optimiser.step() inside the loop?")
except (AttributeError, TypeError):
    print("Make my_model first, for example my_model = MoodNet().to(device)")